# Colab 실행 안내

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flourscent/DLFS_code/blob/master/06_rnns/Autograd_Simple.ipynb)

이 노트북은 로컬 Jupyter와 Google Colab에서 모두 실행할 수 있게 정리했습니다.
학습을 시작할 때는 바로 아래의 **공통 실행 환경 준비** 셀을 먼저 실행하세요.

- Colab에서는 저장소를 `/content/DLFS_code` 아래로 자동 clone합니다.
- 상대 경로가 깨지지 않도록 현재 노트북 폴더로 작업 디렉토리를 이동합니다.
- `lincoln` 패키지가 필요한 장은 import 경로를 자동으로 연결합니다.
- matplotlib 한글 폰트도 가능한 범위에서 자동 설정합니다.


In [ ]:
# 공통 실행 환경 준비
# 이 셀은 Colab과 로컬 Jupyter 사이의 환경 차이를 흡수한다.
# Colab에서는 먼저 저장소를 clone하고, 그 다음 notebook_env를 import해야 한다.
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/flourscent/DLFS_code.git'
REPO_NAME = 'DLFS_code'

if 'google.colab' in sys.modules:
    _bootstrap_repo_root = Path('/content') / REPO_NAME
    if not _bootstrap_repo_root.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(_bootstrap_repo_root)], check=True)
else:
    _bootstrap_repo_root = None
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / 'README.md').exists() and (candidate / 'lincoln').exists():
            _bootstrap_repo_root = candidate
            break
    if _bootstrap_repo_root is None:
        raise FileNotFoundError('DLFS_code 저장소 루트를 찾지 못했습니다.')

if str(_bootstrap_repo_root) not in sys.path:
    sys.path.insert(0, str(_bootstrap_repo_root))

from notebook_env import prepare_notebook_environment

REPO_ROOT = prepare_notebook_environment(
    notebook_dir='06_rnns',
    needs_lincoln=False,
    ensure_mnist=False,
)


# 예제로 본 간단한 자동미분

In [1]:
from typing import Union, List

import numpy as np

np.set_printoptions(precision=4)

In [2]:
a = 3
a.__add__(4)

7

In [3]:
a = np.array([2,3,1,0])

print(a)
print("Addition using '__add__':", a.__add__(4))
print("Addition using '+':", a + 4)

[2 3 1 0]
Addition using '__add__': [6 7 5 4]
Addition using '+': [6 7 5 4]


In [3]:
Numberable = Union[float, int]

def ensure_number(num: Numberable):
    if isinstance(num, NumberWithGrad):
        return num
    else:
        return NumberWithGrad(num)        

class NumberWithGrad(object):
    
    def __init__(self, 
                 num: Numberable,
                 depends_on: List[Numberable] = None,
                 creation_op: str = ''):
        self.num = num
        self.grad = None
        self.depends_on = depends_on or []
        self.creation_op = creation_op

    def __add__(self, 
                other: Numberable):
        return NumberWithGrad(self.num + ensure_number(other).num,
                              depends_on = [self, ensure_number(other)],
                              creation_op = 'add')
    
    def __mul__(self,
                other: Numberable = None):

        return NumberWithGrad(self.num * ensure_number(other).num,
                              depends_on = [self, ensure_number(other)],
                              creation_op = 'mul')
    
    def backward(self, backward_grad: Numberable = None):
        if backward_grad is None: # backward가 처음 호출됨
            self.grad = 1
        else: 
            # 이 부분에서 기울기가 누적됨
            # 기울기 정보가 아직 없다면 backward_grad로 설정
            if self.grad is None:
                self.grad = backward_grad
            # 기울기 정보가 있다면 기존 기울깃값에 backward_grad를 더함
            else:
                self.grad += backward_grad
        
        if self.creation_op == "add":
            # self.grad를 역방향으로 전달함 
            # 둘 중 어느 요소를 증가시켜도 출력이 같은 값만큼 증가함
            self.depends_on[0].backward(self.grad)
            self.depends_on[1].backward(self.grad)    

        if self.creation_op == "mul":

            # 첫 번째 요소에 대한 미분 계산
            new = self.depends_on[1] * self.grad
            # 이 요소에 대한 미분을 역방향으로 전달
            self.depends_on[0].backward(new.num)

            # 두 번째 요소에 대한 미분 계산
            new = self.depends_on[0] * self.grad
            # 이 요소에 대한 미분을 역방향으로 전달
            self.depends_on[1].backward(new.num)

In [4]:
a = NumberWithGrad(3)
b = a * 4
c = b + 3
c.backward()
print(a.grad) # 예상한 값이 출력됨
print(b.grad) # 예상한 값이 출력됨

4
1


In [5]:
a = NumberWithGrad(3)

In [6]:
b = a * 4
c = b + 3
d = (a + 2)
e = c * d 
e.backward() 

In [7]:
a.grad # 예상한 값이 출력됨

35